# 02 — v2 improved recurrent

Objetivo: mejorar v1 con una representación más rica y una arquitectura recurrente más expresiva, manteniendo el mismo protocolo libre de leakage.

Decisiones de v2:
- entrada: **features fusionadas HRV + morfología manual por latido**
- modelo: **BiGRU/BiLSTM apilada + atención temporal**
- desbalance: **focal loss** o **balanced batch**, comparados sobre val

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import tensorflow as tf

from src.utils.reproducibility import set_global_seed
from src.features.sequence_builders import SequenceBuildConfig, build_dual_branch_dataset, concatenate_feature_branches
from src.data.splits import make_fixed_grouped_splits
from src.features.preprocessing import fit_sequence_scaler
from src.training.train import TrainingConfig, fit_model
from src.evaluation.metrics import build_evaluation_bundle
from src.evaluation.reporting import plot_confusion, plot_training_history, measure_inference_time
from src.xai.gradients import saliency_map, integrated_gradients
from src.xai.occlusion import temporal_occlusion_importance
from src.utils.io import save_json, save_joblib

In [ ]:
set_global_seed(42)

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "mit-bih-arrhythmia-database-1.0.0"
OUTPUT_DIR = PROJECT_ROOT / "models" / "v2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_CFG = SequenceBuildConfig(
    data_dir=str(DATA_DIR),
    label_mode="aami_5",
    n_steps=20,
    horizon=1,
    lead=0,
    beat_window_before=90,
    beat_window_after=162,
    exclude_paced_records=False,
)

CANDIDATE_TRAIN_CFGS = [
    TrainingConfig(
        model_version="v2",
        batch_size=128,
        epochs=60,
        learning_rate=1e-3,
        clipnorm=1.0,
        balance_strategy="focal",
        patience=10,
        lr_patience=4,
        checkpoint_path=str(OUTPUT_DIR / "candidate_focal.keras"),
        random_state=42,
        model_kwargs={
            "rnn_type": "gru",
            "units": (64, 32),
            "dropout_rate": 0.25,
            "recurrent_dropout_rate": 0.15,
            "use_attention": True,
            "l2_reg": 1e-4,
        },
    ),
    TrainingConfig(
        model_version="v2",
        batch_size=128,
        epochs=60,
        learning_rate=1e-3,
        clipnorm=1.0,
        balance_strategy="balanced_batch",
        patience=10,
        lr_patience=4,
        checkpoint_path=str(OUTPUT_DIR / "candidate_balanced_batch.keras"),
        random_state=42,
        model_kwargs={
            "rnn_type": "gru",
            "units": (64, 32),
            "dropout_rate": 0.25,
            "recurrent_dropout_rate": 0.15,
            "use_attention": True,
            "l2_reg": 1e-4,
        },
    ),
]

In [ ]:
dataset = build_dual_branch_dataset(DATA_CFG)
X_hrv = dataset["X_hrv"]
X_morph = dataset["X_morph"]
X_fused = concatenate_feature_branches(X_hrv, X_morph)
y = dataset["y"]
groups = dataset["groups"]
class_names = {int(k): v for k, v in dataset["class_names"].items()}

print("X_fused:", X_fused.shape)
print(pd.Series(y).value_counts().sort_index())

In [ ]:
split = make_fixed_grouped_splits(y=y, groups=groups, seed=42)

X_train_raw = X_fused[split.train_idx]
X_val_raw = X_fused[split.val_idx]
X_test_raw = X_fused[split.test_idx]

y_train = y[split.train_idx]
y_val = y[split.val_idx]
y_test = y[split.test_idx]

In [ ]:
scaler_fused = fit_sequence_scaler(X_train_raw, scaler_name="robust")

X_train = scaler_fused.transform(X_train_raw)
X_val = scaler_fused.transform(X_val_raw)
X_test = scaler_fused.transform(X_test_raw)

save_joblib(scaler_fused, OUTPUT_DIR / "scaler_fused.joblib")

## Comparación corta de manejo del desbalance

Aquí comparamos dos opciones razonables dentro de un pipeline temporal:
- focal loss
- balanced batch sampler

Evitamos SMOTE sobre secuencias.

In [ ]:
candidate_rows = []
best_model = None
best_cfg = None
best_val_macro_f1 = -1.0

from src.evaluation.metrics import compute_global_metrics

for cfg in CANDIDATE_TRAIN_CFGS:
    model, history = fit_model(
        x_train=X_train,
        y_train=y_train,
        x_val=X_val,
        y_val=y_val,
        cfg=cfg,
        input_shapes={"sequence_input": X_train.shape[1:]},
        num_classes=len(class_names),
    )
    y_val_prob = model.predict(X_val, verbose=0)
    y_val_pred = np.argmax(y_val_prob, axis=1)
    metrics = compute_global_metrics(y_true=y_val, y_pred=y_val_pred, y_prob=y_val_prob)
    candidate_rows.append({
        "balance_strategy": cfg.balance_strategy,
        **metrics,
    })
    if metrics["macro_f1"] > best_val_macro_f1:
        best_val_macro_f1 = metrics["macro_f1"]
        best_model = model
        best_cfg = cfg
        best_history = history

pd.DataFrame(candidate_rows).sort_values("macro_f1", ascending=False)

In [ ]:
fig = plot_training_history(best_history, title_prefix=f"v2 ({best_cfg.balance_strategy})")
fig

In [ ]:
y_test_prob = best_model.predict(X_test, verbose=0)
y_test_pred = np.argmax(y_test_prob, axis=1)

eval_bundle = build_evaluation_bundle(
    y_true=y_test,
    y_pred=y_test_pred,
    y_prob=y_test_prob,
    class_names=class_names,
)
eval_bundle["selected_balance_strategy"] = best_cfg.balance_strategy
eval_bundle["inference_time"] = measure_inference_time(best_model, X_test[:1], repeats=20)

eval_bundle["global_metrics"]

In [ ]:
plot_confusion(y_test, y_test_pred, class_names, normalize=False)

In [ ]:
plot_confusion(y_test, y_test_pred, class_names, normalize=True)

## Análisis de errores por registro

Un modelo útil no debe fallar siempre en los mismos registros.

In [ ]:
test_meta = dataset["meta"].iloc[split.test_idx].copy()
test_meta["y_true"] = y_test
test_meta["y_pred"] = y_test_pred
test_meta["correct"] = (test_meta["y_true"] == test_meta["y_pred"]).astype(int)

per_record = (
    test_meta.groupby("record_id")
    .agg(samples=("correct", "size"), accuracy=("correct", "mean"))
    .sort_values(["accuracy", "samples"])
)
per_record.head(10)

## XAI más completa

En v2 miramos atribuciones sobre el tensor fusionado HRV + morfología.

In [ ]:
sample_idx = 0
sample = X_test[sample_idx:sample_idx+1]

sample_saliency = saliency_map(best_model, sample)
sample_ig = integrated_gradients(best_model, sample, steps=32)
sample_occ = temporal_occlusion_importance(best_model, sample, window=1)

print("Saliency shape:", sample_saliency.shape)
print("IG shape:", sample_ig.shape)
print("Occlusion shape:", sample_occ.shape)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].imshow(np.abs(sample_saliency[0]).T, aspect="auto", origin="lower")
axes[0].set_title("Saliency | fusionado")
axes[0].set_xlabel("Timestep")
axes[0].set_ylabel("Feature")

axes[1].imshow(np.abs(sample_ig[0]).T, aspect="auto", origin="lower")
axes[1].set_title("Integrated gradients | fusionado")
axes[1].set_xlabel("Timestep")

axes[2].bar(np.arange(len(sample_occ)), sample_occ)
axes[2].set_title("Occlusion por timestep")
axes[2].set_xlabel("Timestep")
axes[2].set_ylabel("Caída de probabilidad")

plt.tight_layout()
plt.show()

In [ ]:
metadata = {
    "model_version": "v2",
    "task": "next_beat_aami5_grouped",
    "class_names": class_names,
    "input_shapes": {"sequence_input": list(X_train.shape[1:])},
    "feature_names": dataset["feature_names_hrv"] + dataset["feature_names_morph"],
    "split_protocol": "grouped_stratified_record_split",
    "balance_strategy": best_cfg.balance_strategy,
    "loss_name": best_cfg.loss_name,
    "seed": best_cfg.random_state,
    "data_config": dataset["config"],
}

best_model.save(OUTPUT_DIR / "model.keras")
save_json(metadata, OUTPUT_DIR / "metadata.json")
save_json(eval_bundle, OUTPUT_DIR / "metrics_test.json")

## Cierre v2

La mejora solo es válida si sube el macro-F1 y mantiene una degradación controlada por registro, no si solo mejora accuracy global.